# 🚀 Modern Local LLM Fine-Tuning & Comprehensive Evals with Apple MLX
### *A Production-Grade Guide to LoRA/QLoRA and Multi-Pillar Evaluations on Apple Silicon*

Welcome! In this tutorial, you will learn how to:
1. **Leverage Apple Silicon's Unified Memory Architecture** using Apple's native **MLX** framework.
2. **Fine-tune an Instruct Model (Qwen 2.5, 3B–72B)** for high-reliability **JSON Tool Calling** using LoRA / QLoRA.
3. **Apply Prompt Masking** so loss backpropagation only updates assistant response tokens.
4. **Implement a 4-Pillar Evaluation Suite**:
   - **Pillar 1: Intrinsic Metrics** (Cross-Entropy Loss & Test Perplexity)
   - **Pillar 2: Deterministic Metrics** (Strict JSON parseability, Pydantic schema adherence, parameter exact match)
   - **Pillar 3: Qualitative Analysis** (Base vs. LoRA side-by-side behavioral review)
   - **Pillar 4: Systems Profiling** (Metal unified memory & token throughput)
5. **Fuse LoRA Adapters** into standalone weights for zero-overhead local deployment.

---

In [ ]:
import sys
import os
import json
import time
import math
from pathlib import Path

import mlx.core as mx
import mlx.nn as nn
import mlx_lm
import matplotlib.pyplot as plt
from rich.console import Console
from rich.table import Table

console = Console()

# Verify Apple Silicon Hardware & Metal Acceleration
print(f"Python Version    : {sys.version.split()[0]}")
print(f"MLX Version       : {mx.__version__}")
print(f"Default Device    : {mx.default_device()}")
print(f"Metal Acceleration: {mx.metal.is_available()}")

# Peak memory check
mx.reset_peak_memory()
print(f"Initial Memory    : {mx.get_active_memory() / (1024**2):.2f} MB")

## 1. Defining the Operational Tool Schema (Pydantic v2)

Base instruction models frequently struggle with strict JSON generation:
- They wrap outputs in Markdown code blocks (` ```json ... ``` `).
- They hallucinate parameters or use invalid enum variants (e.g. `"prod"` instead of `"production"`).
- They add conversational filler before or after the JSON.

To test this deterministically, we define 4 infrastructure operations using **Pydantic**:
1. `deploy_service` (service, version, environment, replicas, channels)
2. `restart_pod` (pod_name, region, force, reason)
3. `rollback_deployment` (deployment_id, target_tag, drain_traffic)
4. `scale_cluster` (cluster_name, node_count, auto_scale, instance_type)

In [ ]:
from src.schema import (
    SYSTEM_PROMPT,
    DeployServiceParams,
    RestartPodParams,
    RollbackDeploymentParams,
    ScaleClusterParams,
    parse_and_validate
)

# Test our parser against typical LLM output variations
test_output_clean = '{"tool": "deploy_service", "parameters": {"service": "auth-api", "version": "v2.1.0", "environment": "production", "replicas": 3}}'
test_output_markdown = 'Sure! Here is the JSON:\n```json\n{"tool": "restart_pod", "parameters": {"pod_name": "postgres-0", "region": "us-west-2", "force": true, "reason": "OOM"}}\n```'

res_clean = parse_and_validate(test_output_clean)
res_md = parse_and_validate(test_output_markdown)

print("Clean JSON Parse Result:")
print(f"  Pure JSON: {res_clean['is_pure_json']} | Schema Valid: {res_clean['is_schema_valid']}")

print("\nMarkdown JSON Parse Result:")
print(f"  Pure JSON: {res_md['is_pure_json']} (Fails strict pure check) | Schema Valid: {res_md['is_schema_valid']}")

## 2. Dataset Engineering & Prompt Masking

When fine-tuning instruction models, **Prompt Masking** is essential:
- Without prompt masking, the model calculates loss over the system prompt and user instructions, wasting gradient updates memorizing the input format.
- With `mask_prompt: true`, loss is **only** computed on the target tokens (the assistant's JSON output).

MLX expects datasets in standard ChatML/OpenAI JSONL format:
```json
{
  "messages": [
    {"role": "system", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."}
  ]
}
```

In [ ]:
# Inspect generated dataset splits
with open("data/train.jsonl", "r") as f:
    train_lines = f.readlines()
with open("data/valid.jsonl", "r") as f:
    valid_lines = f.readlines()
with open("data/test.jsonl", "r") as f:
    test_lines = f.readlines()

print(f"Training Samples  : {len(train_lines)}")
print(f"Validation Samples: {len(valid_lines)}")
print(f"Holdout Test Samples: {len(test_lines)}")

# Preview one sample
sample = json.loads(train_lines[0])
print("\n--- User Request Sample ---")
print(sample["messages"][1]["content"])
print("\n--- Expected Assistant JSON ---")
print(sample["messages"][2]["content"])

## 3. LoRA & QLoRA Fine-Tuning on Apple Silicon

### Why LoRA?
Instead of modifying all 3 billion weights, **Low-Rank Adaptation (LoRA)** freezes the base model and injects trainable rank decomposition matrices:
$$W' = W + \frac{\alpha}{r} (B \times A)$$

For `Qwen2.5-3B-Instruct-4bit`:
- Base parameters: **3.08 Billion** (quantized to 4 bits)
- Trainable parameters: **6.65 Million** (~0.21% of model)
- Adapter size on disk: **~25 MB** (instead of 6 GB)
- Memory required: **< 10 GB** during backpropagation!

In [ ]:
from src.models import PRESETS
from src.train import run_training

# Choose 3b, 7b, 14b, 32b, or 72b. Start with a short run for larger models.
# Larger presets use smaller batches and gradient checkpointing.
PRESET = "3b"
profile = PRESETS[PRESET]
adapter_file = Path(profile.adapter_path) / "adapters.safetensors"
print(f"Model: {profile.model} | Artifacts: {profile.output_dir}")

# Run training (or inspect existing artifacts for this model)
if not adapter_file.exists():
    print("Executing fine-tuning...")
    run_training(preset=PRESET, iters_override=80)
else:
    print(f"Found existing fine-tuned LoRA adapters in {profile.adapter_path}!")
    adapter_size_mb = adapter_file.stat().st_size / (1024**2)
    print(f"Adapter weight file size: {adapter_size_mb:.2f} MB")

## 4. The 4-Pillar Comprehensive Evaluation Suite

A modern MLX evaluation is not a single prompt inspection. We execute a rigorous 4-pillar benchmark:

| Pillar | Focus | What It Measures |
| :--- | :--- | :--- |
| **1. Intrinsic Metrics** | Statistical Fit | Cross-Entropy Loss & Perplexity on unseen holdout test set |
| **2. Deterministic Evals** | Functional Correctness | Pure JSON rate, Pydantic schema validity, Tool accuracy, Parameter Exact Match |
| **3. Qualitative Review** | Error Taxonomy | Side-by-side comparison of Base vs. LoRA completions |
| **4. Systems & Hardware** | Metal Efficiency | Generation throughput (tok/s), Peak Unified Memory (MB) |

In [ ]:
from src.evaluate import run_comprehensive_evaluation

# Run the 4-Pillar evaluation on holdout test samples
report = run_comprehensive_evaluation(
    preset=PRESET,
    num_eval_samples=20
)

In [ ]:
# Display Evaluation Comparison Chart
from IPython.display import Image, display

if (profile.output_dir / "eval_comparison.png").exists():
    display(Image(filename=str(profile.output_dir / "eval_comparison.png")))
else:
    print("eval_comparison.png not found. Run evaluation first.")

## 5. Inference Benchmarking & Model Weight Fusion

### Dynamic LoRA vs. Weight Fusion
- **Dynamic LoRA**: At inference time, the model executes the base layer, computes the adapter branch, and adds them. This incurs a small latency penalty.
- **Model Fusion (`mlx_lm.fuse`)**: Fuses $W + \frac{\alpha}{r}(B \times A)$ directly into a new standalone model weights file.
  - Zero runtime overhead: runs at full base-model speed (~140 tok/s).
  - Standalone: No adapter files needed at runtime.
  - Ready for local API serving via `mlx_lm.server`.

In [ ]:
from src.benchmark import run_benchmark_suite

# Run benchmark across Base vs LoRA
run_benchmark_suite(
    preset=PRESET
)

print("\nTo serve your fine-tuned model as an OpenAI-compatible API on Apple Silicon:")
print(f"  uv run python main.py fuse --preset {PRESET}")
print(f"  uv run python main.py serve --preset {PRESET} --port 8080")